# 🔮 Walk-Forward Predictions - Out-of-Sample

## 🎯 Objetivo

Gerar previsões **estritamente out-of-sample** do modelo XGBoost ajustado para o período de backtest (2022-2024), usando validação temporal walk-forward com janela expansiva.

---

## 📊 Especificações

### Dataset:
* **workspace.gold.fii_features_v1** (23 features)
* Target: **target_7d** (superou IFIX em 7 pregões?)

### Modelo:
* **XGBoost ajustado** (hiperparâmetros do notebook 38)
* Configuração: `FINAL_HYPERPARAMETERS_REVISED`

### Período de Backtest:
* **Janeiro 2022 até Dezembro 2024** (~3 anos)
* Treino inicial: 2020-03 até 2021-12

### Walk-Forward:
* **Janela expansiva** (usa todos dados anteriores)
* **Gap obrigatório**: 7 pregões entre treino e previsão
* **Retreinamento**: Trimestral (mar, jun, set, dez)

### Rebalanceamento:
* **Frequência**: A cada 7 pregões
* Calendário comum entre FIIs e IFIX

### Regra de Execução:
* **Sinal gerado**: Data t (após fechamento)
* **Execução**: Data t+1 (close)
* Previsões geradas apenas nas datas de rebalanceamento

---

## 📦 Output

Tabela intermediária: **workspace.gold.fii_walk_forward_predictions**

Colunas:
* `data_sinal`: Data em que o sinal foi gerado
* `data_execucao`: Data de execução (t+1)
* `ticker`: FII
* `probabilidade`: P(target_7d = 1)
* `ranking`: Posição do FII (1 = maior prob)
* `target_7d`: Target real observado
* `modelo_id`: Identificador do modelo usado
* `treino_ate`: Última data de treino

---

## ✅ Garantias

* ✅ Todas as previsões são out-of-sample
* ✅ Gap de 7 pregões respeitado
* ✅ Nenhum dado futuro no treinamento
* ✅ Retreinamento trimestral
* ✅ Auditável e reproduzível

In [0]:
# Instalar dependências
%pip install xgboost scikit-learn
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Seed para reprodutibilidade
SEED = 42
np.random.seed(SEED)

print("✅ Imports carregados")
print(f"Data de execução: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
print("=" * 80)
print("📊 CARREGANDO DADOS")
print("=" * 80)

# Carregar Gold V1
df = spark.table("workspace.gold.fii_features_v1").toPandas()

print(f"\n✅ Gold V1 carregada: {df.shape[0]} registros, {df.shape[1]} colunas")

# Converter date
df['date'] = pd.to_datetime(df['date'])

# Ordenar
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"Período: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Tickers: {sorted(df['ticker'].unique())}")

# Colunas não-features
non_feature_cols = ['ticker', 'date', 'target_7d', 'target_alpha_7d']

# Features
features = [col for col in df.columns if col not in non_feature_cols]

print(f"\nFeatures: {len(features)}")
print(f"Target: target_7d")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🔧 HIPERPARÂMETROS DO MODELO AJUSTADO")
print("=" * 80)

# Hiperparâmetros do notebook 38_hyperparameter_tuning (FINAL_HYPERPARAMETERS_REVISED)
FINAL_HYPERPARAMETERS = {
    'n_estimators': 150,
    'max_depth': 6,
    'learning_rate': 0.07,
    'min_child_weight': 1,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'gamma': 0.0,
    'reg_alpha': 0.01,
    'reg_lambda': 1.5,
    'early_stopping_rounds': 20,
    'random_state': SEED,
    'eval_metric': 'auc',
    'verbosity': 0
}

print("\nConfiguração XGBoost AJUSTADO (Tuned):")
for k, v in FINAL_HYPERPARAMETERS.items():
    print(f"  {k}: {v}")

print("\n✅ Hiperparâmetros carregados")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📅 DEFINIÇÃO DAS DATAS DE REBALANCEAMENTO")
print("=" * 80)

# Período do backtest
backtest_start = pd.to_datetime('2022-01-01')
backtest_end = pd.to_datetime('2024-12-31')

# Datas únicas disponíveis no período de backtest
backtest_dates = df[(df['date'] >= backtest_start) & (df['date'] <= backtest_end)]['date'].drop_duplicates().sort_values().reset_index(drop=True)

print(f"\nPeríodo de backtest: {backtest_start.date()} a {backtest_end.date()}")
print(f"Datas de pregão disponíveis: {len(backtest_dates)}")

# Definir datas de rebalanceamento: a cada 7 pregões
rebalance_dates = []
for i in range(0, len(backtest_dates), 7):
    rebalance_dates.append(backtest_dates.iloc[i])

rebalance_dates = pd.Series(rebalance_dates)

print(f"\nDatas de rebalanceamento: {len(rebalance_dates)}")
print(f"Primeira data: {rebalance_dates.iloc[0].date()}")
print(f"Última data: {rebalance_dates.iloc[-1].date()}")

# Mostrar primeiras e últimas 5 datas
print(f"\nPrimeiras 5 datas de rebalanceamento:")
for i in range(min(5, len(rebalance_dates))):
    print(f"  {i+1}. {rebalance_dates.iloc[i].date()}")

print(f"\nÚltimas 5 datas de rebalanceamento:")
for i in range(max(0, len(rebalance_dates)-5), len(rebalance_dates)):
    print(f"  {i+1}. {rebalance_dates.iloc[i].date()}")

print("\n✅ Datas de rebalanceamento definidas")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🔄 SCHEDULE DE RETREINAMENTO (TRIMESTRAL)")
print("=" * 80)

# Treino inicial: 2020-03 até 2021-12
initial_train_start = pd.to_datetime('2020-03-01')
initial_train_end = pd.to_datetime('2021-12-31')

# Retreinos trimestrais: final de cada trimestre
retrain_schedule = [
    {'nome': 'Treino Inicial', 'treino_ate': '2021-12-31', 'valido_ate': '2022-03-31'},
    {'nome': 'Q1 2022', 'treino_ate': '2022-03-31', 'valido_ate': '2022-06-30'},
    {'nome': 'Q2 2022', 'treino_ate': '2022-06-30', 'valido_ate': '2022-09-30'},
    {'nome': 'Q3 2022', 'treino_ate': '2022-09-30', 'valido_ate': '2022-12-31'},
    {'nome': 'Q4 2022', 'treino_ate': '2022-12-31', 'valido_ate': '2023-03-31'},
    {'nome': 'Q1 2023', 'treino_ate': '2023-03-31', 'valido_ate': '2023-06-30'},
    {'nome': 'Q2 2023', 'treino_ate': '2023-06-30', 'valido_ate': '2023-09-30'},
    {'nome': 'Q3 2023', 'treino_ate': '2023-09-30', 'valido_ate': '2023-12-31'},
    {'nome': 'Q4 2023', 'treino_ate': '2023-12-31', 'valido_ate': '2024-03-31'},
    {'nome': 'Q1 2024', 'treino_ate': '2024-03-31', 'valido_ate': '2024-06-30'},
    {'nome': 'Q2 2024', 'treino_ate': '2024-06-30', 'valido_ate': '2024-09-30'},
    {'nome': 'Q3 2024', 'treino_ate': '2024-09-30', 'valido_ate': '2024-12-31'},
]

print(f"\nTotal de modelos a treinar: {len(retrain_schedule)}\n")

for i, schedule in enumerate(retrain_schedule, 1):
    print(f"{i:2d}. {schedule['nome']:<20} | Treino até: {schedule['treino_ate']} | Válido até: {schedule['valido_ate']}")

print("\n✅ Schedule de retreinamento definido")
print("\n" + "=" * 80)

In [0]:
def generate_walk_forward_predictions(df, features, rebalance_dates, retrain_schedule, gap_days=7):
    """
    Gera previsões out-of-sample usando walk-forward com gap de 7 pregões.
    
    Para cada data de rebalanceamento:
    1. Determinar qual modelo usar (baseado no schedule de retreinamento)
    2. Treinar modelo com dados até treino_ate - gap
    3. Gerar previsões para a data de rebalanceamento (data_sinal)
    4. Data de execução = próximo pregão após data_sinal
    
    Retorna:
        DataFrame com previsões, rankings, targets, datas de sinal e execução
    """
    predictions = []
    all_dates_sorted = df['date'].drop_duplicates().sort_values().reset_index(drop=True)
    
    for idx_rebal, data_sinal in enumerate(rebalance_dates, 1):
        # Determinar qual modelo usar
        modelo_info = None
        for schedule in retrain_schedule:
            valido_ate = pd.to_datetime(schedule['valido_ate'])
            if data_sinal <= valido_ate:
                modelo_info = schedule
                break
        
        if modelo_info is None:
            print(f"\n⚠️  Data {data_sinal.date()} fora do schedule de retreinamento. Pulando.")
            continue
        
        treino_ate = pd.to_datetime(modelo_info['treino_ate'])
        modelo_id = modelo_info['nome']
        
        # Aplicar gap: treinar até (treino_ate - gap_days pregões)
        dates_until_train_end = all_dates_sorted[all_dates_sorted <= treino_ate]
        if len(dates_until_train_end) < gap_days:
            print(f"\n⚠️  Dados insuficientes para gap na data {data_sinal.date()}. Pulando.")
            continue
        
        train_end_with_gap = dates_until_train_end.iloc[-gap_days - 1]  # -gap-1 para ter gap de 7 pregões
        
        # Máscara de treino: desde início até train_end_with_gap
        train_mask = (df['date'] <= train_end_with_gap)
        
        # Máscara de previsão: apenas data_sinal
        pred_mask = (df['date'] == data_sinal)
        
        if train_mask.sum() == 0 or pred_mask.sum() == 0:
            continue
        
        # Preparar dados de treino
        X_train = df.loc[train_mask, features]
        y_train = df.loc[train_mask, 'target_7d']
        
        # Preparar dados de previsão
        X_pred = df.loc[pred_mask, features]
        tickers_pred = df.loc[pred_mask, 'ticker'].values
        targets_pred = df.loc[pred_mask, 'target_7d'].values
        
        # Treinar modelo (remover early_stopping_rounds para backtest)
        params_backtest = FINAL_HYPERPARAMETERS.copy()
        params_backtest.pop('early_stopping_rounds', None)  # Remover early stopping
        
        model = xgb.XGBClassifier(**params_backtest)
        model.fit(X_train, y_train)
        
        # Gerar previsões
        probs = model.predict_proba(X_pred)[:, 1]
        
        # Data de execução: próximo pregão após data_sinal
        future_dates = all_dates_sorted[all_dates_sorted > data_sinal]
        if len(future_dates) == 0:
            data_execucao = None
        else:
            data_execucao = future_dates.iloc[0]
        
        # Calcular ranking (1 = maior probabilidade)
        ranking_dict = {}
        sorted_indices = np.argsort(probs)[::-1]  # maior para menor
        for rank, idx in enumerate(sorted_indices, 1):
            ranking_dict[tickers_pred[idx]] = rank
        
        # Armazenar previsões
        for i, ticker in enumerate(tickers_pred):
            predictions.append({
                'data_sinal': data_sinal,
                'data_execucao': data_execucao,
                'ticker': ticker,
                'probabilidade': probs[i],
                'ranking': ranking_dict[ticker],
                'target_7d': targets_pred[i],
                'modelo_id': modelo_id,
                'treino_ate': train_end_with_gap
            })
        
        # Progress
        if idx_rebal % 10 == 0:
            print(f"  Progress: {idx_rebal}/{len(rebalance_dates)} datas de rebalanceamento processadas")
    
    return pd.DataFrame(predictions)

print("✅ Função generate_walk_forward_predictions criada")

In [0]:
print("=" * 80)
print("🔮 GERANDO PREVISÕES OUT-OF-SAMPLE")
print("=" * 80)

import time
start_time = time.time()

print(f"\nIniciando walk-forward...")
print(f"Datas de rebalanceamento: {len(rebalance_dates)}")
print(f"Gap: 7 pregões\n")

predictions_df = generate_walk_forward_predictions(
    df=df,
    features=features,
    rebalance_dates=rebalance_dates,
    retrain_schedule=retrain_schedule,
    gap_days=7
)

elapsed_time = time.time() - start_time

print(f"\n✅ Previsões geradas em {elapsed_time/60:.1f} minutos")
print(f"\nTotal de previsões: {len(predictions_df)}")
print(f"Datas únicas de sinal: {predictions_df['data_sinal'].nunique()}")
print(f"Tickers: {sorted(predictions_df['ticker'].unique())}")
print(f"Modelos usados: {sorted(predictions_df['modelo_id'].unique())}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("✅ AUDITORIA: GARANTIAS OUT-OF-SAMPLE")
print("=" * 80)

# 1. Verificar que todas as previsões são out-of-sample
print("\n1️⃣ Verificando que data_sinal > treino_ate (out-of-sample):")

violations_oos = predictions_df[predictions_df['data_sinal'] <= predictions_df['treino_ate']]

if len(violations_oos) == 0:
    print("   ✅ PASSOU: Todas as previsões são out-of-sample")
else:
    print(f"   ❌ FALHOU: {len(violations_oos)} previsões com violação!")
    display(violations_oos.head())

# 2. Verificar gap de 7 pregões
print("\n2️⃣ Verificando gap de 7 pregões entre treino_ate e data_sinal:")

all_dates_sorted = df['date'].drop_duplicates().sort_values().reset_index(drop=True)

def count_business_days_between(date1, date2, all_dates):
    """Conta pregões entre duas datas"""
    mask = (all_dates > date1) & (all_dates < date2)
    return mask.sum()

gaps = []
for _, row in predictions_df.drop_duplicates('data_sinal').iterrows():
    gap_count = count_business_days_between(row['treino_ate'], row['data_sinal'], all_dates_sorted)
    gaps.append(gap_count)

min_gap = min(gaps)
max_gap = max(gaps)
median_gap = np.median(gaps)

print(f"   Gap mínimo: {min_gap} pregões")
print(f"   Gap máximo: {max_gap} pregões")
print(f"   Gap mediano: {median_gap} pregões")

if min_gap >= 7:
    print("   ✅ PASSOU: Gap de pelo menos 7 pregões respeitado")
else:
    print(f"   ⚠️  ATENÇÃO: Gap mínimo é {min_gap} pregões (esperado ≥ 7)")

# 3. Verificar data_execucao = data_sinal + 1 pregão
print("\n3️⃣ Verificando que data_execucao = próximo pregão após data_sinal:")

violations_exec = 0
for _, row in predictions_df.drop_duplicates('data_sinal').iterrows():
    next_date_expected = all_dates_sorted[all_dates_sorted > row['data_sinal']].iloc[0]
    if row['data_execucao'] != next_date_expected:
        violations_exec += 1

if violations_exec == 0:
    print("   ✅ PASSOU: Todas as datas de execução são corretas")
else:
    print(f"   ❌ FALHOU: {violations_exec} datas de execução incorretas")

# 4. Verificar que não há dados futuros
print("\n4️⃣ Verificando que não há dados futuros no treino:")

# Treino_ate deve ser sempre < data_sinal
if (predictions_df['treino_ate'] < predictions_df['data_sinal']).all():
    print("   ✅ PASSOU: Nenhum dado futuro usado no treino")
else:
    print("   ❌ FALHOU: Dados futuros detectados no treino!")

print("\n" + "=" * 80)
print("✅ AUDITORIA CONCLUÍDA")
print("=" * 80)

In [0]:
print("=" * 80)
print("📊 ESTATÍSTICAS DAS PREVISÕES")
print("=" * 80)

print("\n📅 Período:")
print(f"  Primeira data de sinal: {predictions_df['data_sinal'].min().date()}")
print(f"  Última data de sinal: {predictions_df['data_sinal'].max().date()}")
print(f"  Primeira data de execução: {predictions_df['data_execucao'].min().date()}")
print(f"  Última data de execução: {predictions_df['data_execucao'].max().date()}")

print("\n📊 Contagens:")
print(f"  Total de previsões: {len(predictions_df)}")
print(f"  Datas de rebalanceamento: {predictions_df['data_sinal'].nunique()}")
print(f"  Tickers: {predictions_df['ticker'].nunique()}")
print(f"  Modelos usados: {predictions_df['modelo_id'].nunique()}")

print("\n🎯 Previsões por ticker:")
for ticker in sorted(predictions_df['ticker'].unique()):
    count = (predictions_df['ticker'] == ticker).sum()
    print(f"  {ticker}: {count}")

print("\n🤖 Previsões por modelo:")
for modelo in sorted(predictions_df['modelo_id'].unique()):
    count = (predictions_df['modelo_id'] == modelo).sum()
    pct = 100 * count / len(predictions_df)
    print(f"  {modelo:<20}: {count:4d} ({pct:5.1f}%)")

print("\n🎲 Estatísticas das probabilidades:")
print(predictions_df['probabilidade'].describe())

print("\n🎯 Distribuição dos targets (target_7d):")
target_dist = predictions_df['target_7d'].value_counts()
for target, count in target_dist.items():
    pct = 100 * count / len(predictions_df)
    print(f"  target_7d = {target}: {count} ({pct:.1f}%)")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🔍 AMOSTRA AUDITÁVEL DAS PREVISÕES")
print("=" * 80)

print("\n📅 Primeiras 10 previsões (ordenadas por data_sinal):")

# Preparar amostra
amostra = predictions_df.sort_values(['data_sinal', 'ranking']).head(10).copy()

# Formatar datas para visualização
amostra['data_sinal'] = amostra['data_sinal'].dt.date
amostra['data_execucao'] = amostra['data_execucao'].dt.date
amostra['treino_ate'] = amostra['treino_ate'].dt.date

# Formatar probabilidade
amostra['probabilidade'] = amostra['probabilidade'].round(4)

# Selecionar colunas relevantes
cols_display = ['data_sinal', 'data_execucao', 'ticker', 'probabilidade', 'ranking', 'target_7d', 'modelo_id', 'treino_ate']

display(amostra[cols_display])

print("\n📅 Últimas 10 previsões (ordenadas por data_sinal):")

# Preparar amostra
amostra_fim = predictions_df.sort_values(['data_sinal', 'ranking']).tail(10).copy()

# Formatar datas para visualização
amostra_fim['data_sinal'] = amostra_fim['data_sinal'].dt.date
amostra_fim['data_execucao'] = amostra_fim['data_execucao'].dt.date
amostra_fim['treino_ate'] = amostra_fim['treino_ate'].dt.date

# Formatar probabilidade
amostra_fim['probabilidade'] = amostra_fim['probabilidade'].round(4)

display(amostra_fim[cols_display])

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🗓️ EXEMPLOS DE PREVISÕES POR ANO")
print("=" * 80)

# Adicionar coluna de ano
predictions_df['ano'] = pd.to_datetime(predictions_df['data_sinal']).dt.year

for ano in sorted(predictions_df['ano'].unique()):
    print(f"\n📅 Ano {ano}:")
    
    # Primeira data do ano
    primeira_data = predictions_df[predictions_df['ano'] == ano]['data_sinal'].min()
    amostra_ano = predictions_df[
        (predictions_df['data_sinal'] == primeira_data)
    ].sort_values('ranking').copy()
    
    print(f"\n  Primeira data de rebalanceamento: {primeira_data.date()}")
    print(f"  Previsões:")
    
    for _, row in amostra_ano.iterrows():
        prob_pct = row['probabilidade'] * 100
        target_symbol = "✅" if row['target_7d'] == 1 else "❌"
        print(f"    Rank {row['ranking']}: {row['ticker']} - Prob: {prob_pct:.2f}% - Target: {row['target_7d']} {target_symbol}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📦 SALVANDO PREVISÕES EM TABELA INTERMEDIÁRIA")
print("=" * 80)

# Nome da tabela
table_name = "workspace.gold.fii_walk_forward_predictions"

print(f"\nTabela destino: {table_name}")

# Preparar DataFrame para salvar
predictions_to_save = predictions_df.copy()

# Remover coluna temporária 'ano' se existir
if 'ano' in predictions_to_save.columns:
    predictions_to_save = predictions_to_save.drop(columns=['ano'])

# Converter para Spark DataFrame
spark_df = spark.createDataFrame(predictions_to_save)

# Salvar como tabela Delta (substituir se existir)
spark_df.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✅ Tabela salva com sucesso!")
print(f"   Total de registros: {len(predictions_to_save)}")
print(f"   Colunas: {list(predictions_to_save.columns)}")

# Verificar tabela salva
verify_df = spark.table(table_name)
print(f"\n✅ Verificação: {verify_df.count()} registros na tabela")

print("\n" + "=" * 80)

In [0]:
print("="*80)
print("🎯 SUMÁRIO FINAL - NOTEBOOK 39A")
print("="*80)

print("\n📋 DADOS DE SAÍDA:")
print(f"  Tabela: workspace.gold.fii_walk_forward_predictions")
print(f"  Total de previsões: {len(predictions_df)}")
print(f"  Datas de rebalanceamento: {predictions_df['data_sinal'].nunique()}")
print(f"  Período: {predictions_df['data_sinal'].min().date()} a {predictions_df['data_sinal'].max().date()}")

print("\n📅 FREQUÊNCIA:")
print(f"  Rebalanceamento: a cada 7 pregões")
print(f"  Retreinamento: Trimestral (12 modelos)")

print("\n🎯 TICKERS:")
for ticker in sorted(predictions_df['ticker'].unique()):
    count = (predictions_df['ticker'] == ticker).sum()
    print(f"  {ticker}: {count} previsões")

print("\n🤖 MODELOS:")
for modelo in sorted(predictions_df['modelo_id'].unique()):
    count = (predictions_df['modelo_id'] == modelo).sum()
    dates = predictions_df[predictions_df['modelo_id'] == modelo]['data_sinal'].unique()
    print(f"  {modelo}: {len(dates)} datas")

print("\n✅ VALIDAÇÕES APROVADAS:")
print("  ✅ Todas as previsões são out-of-sample")
print("  ✅ Gap de 7 pregões respeitado")
print("  ✅ Data de execução = próximo pregão após sinal")
print("  ✅ Nenhum dado futuro usado no treinamento")
print("  ✅ Janela expansiva aplicada")
print("  ✅ Retreinamento trimestral")

print("\n📊 SCHEMA DA TABELA:")
print("\n  Colunas:")
for col in predictions_df.columns:
    dtype = predictions_df[col].dtype
    print(f"    - {col:<20} ({dtype})")

print("\n🚀 PRÓXIMOS PASSOS:")
print("  1. Implementar notebook 39B_backtest")
print("  2. Consumir workspace.gold.fii_walk_forward_predictions")
print("  3. Construir estratégias de carteira")
print("  4. Calcular retornos e métricas financeiras")
print("  5. Comparar com IFIX e benchmarks")

print("\n" + "="*80)
print("✅ NOTEBOOK 39A CONCLUÍDO COM SUCESSO")
print("="*80)